In [0]:
import pyspark.sql.functions as F

catalog_name = "f1_dev"
target_table = f"{catalog_name}.silver.dim_constructors"



In [0]:
# 1. Read raw constructors table
df_constructors_raw = spark.table(f"{catalog_name}.bronze.constructors")



In [0]:
# 2. Transform & Cleanse
df_constructors_silver = (
    df_constructors_raw
    .replace(r"\N", None)
    .select(
        F.col("constructorId").cast("integer").alias("constructor_id"),
        F.col("constructorRef").alias("constructor_ref"),
        F.col("name").alias("constructor_name"),
        F.col("nationality")
    )
)



In [0]:
# 3. Write as Delta table in Silver schema
df_constructors_silver.write.mode("overwrite").saveAsTable(target_table)



In [0]:
# 4. Set Unity Catalog Governance Tags
spark.sql(f"""
    ALTER TABLE {target_table} SET TAGS (
        'layer' = 'silver',
        'table_type' = 'dimension',
        'contains_pii' = 'false'
    )
""")

print(f"✓ Successfully built and governed: {target_table}")